In [32]:
### Import libraries ###
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.python.tools.saved_model_aot_compile import freeze_model
from tf_keras.src.losses import mean_absolute_percentage_error
import matplotlib.pyplot as plt
import math

from transformers.models.flava.modeling_flava import FlavaSelfOutput

import helper_func as hf
import os
import time
import random

from scipy.stats import kendalltau

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error, mean_absolute_error

import torch.nn as nn
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset, Dataset, Subset
from torch import nn, optim
import torch.nn.functional as F
from torchinfo import summary
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader, TensorDataset

In [33]:
SEED = 101
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
TARGET_VARIABLE = "Mole fraction"

In [34]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
X_val = pd.read_csv("X_val.csv")
y_train = pd.read_csv("y_train.csv")
y_test = pd.read_csv("y_test.csv")
y_val = pd.read_csv("y_val.csv")

KeyboardInterrupt: 

In [ ]:
X_train.drop(
    columns=["Component 1", "Component 2", "Smiles 1", "Smiles 2", "mol1", "mol2"],
    inplace=True
)
X_test.drop(
    columns=["Component 1", "Component 2", "Smiles 1", "Smiles 2", "mol1", "mol2"],
    inplace=True
)
X_val.drop(
    columns=["Component 1", "Component 2", "Smiles 1", "Smiles 2", "mol1", "mol2"],
    inplace=True
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled   = scaler_X.transform(X_val)
X_test_scaled  = scaler_X.transform(X_test)

# Convert to tensors
def to_loader(X, y, batch_size=1024, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(y), dtype=torch.float32).squeeze().unsqueeze(1)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train_scaled, y_train, shuffle=True)
val_loader   = to_loader(X_val_scaled, y_val)
test_loader  = to_loader(X_test_scaled, y_test)

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Hardtanh(min_val=0.0, max_val=1.0)
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
model = MLPRegressor(input_dim)

In [ ]:
model = MLPRegressor(X_train.shape[1], dropout=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

In [ ]:
# Training loop
n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        

    model.eval()
    with torch.no_grad():
        val_losses = [
            criterion(model(X_b.to(device)), y_b.to(device)).item()
            for X_b, y_b in val_loader
        ]
    val_loss = np.mean(val_losses)
    scheduler.step(np.mean(val_losses))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} — Val MSE: {val_loss:.4f}")


Epoch 10/100 — Val MSE: 0.0463
Epoch 20/100 — Val MSE: 0.0314
Epoch 30/100 — Val MSE: 0.0279
Epoch 40/100 — Val MSE: 0.0277
Epoch 50/100 — Val MSE: 0.0280


KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    y_pred = model(X_test_tensor).cpu().numpy().flatten()

In [ ]:
def evaluate_model(y_test, y_pred, show_plot=False, save_file=False, save_path="./results.json"):
    """Evaluate model performance given true and predicted values."""
    y_test = np.array(y_test).flatten()
    y_pred = np.array(y_pred).flatten()

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    tau = kendalltau(y_test, y_pred).correlation

    # Bootstrap CI
    rng = np.random.default_rng(101)
    mae_scores, mse_scores, r2_scores, tau_scores = [], [], [], []

    for _ in range(1000):
        idx = rng.integers(0, len(y_test), len(y_test))
        mae_scores.append(mean_absolute_error(y_test[idx], y_pred[idx]))
        mse_scores.append(mean_squared_error(y_test[idx], y_pred[idx]))
        r2_scores.append(r2_score(y_test[idx], y_pred[idx]))
        tau_scores.append(kendalltau(y_test[idx], y_pred[idx]).correlation)

    mae_ci  = np.percentile(mae_scores, [2.5, 97.5])
    mse_ci  = np.percentile(mse_scores, [2.5, 97.5])
    r2_ci   = np.percentile(r2_scores,  [2.5, 97.5])
    tau_ci  = np.percentile(tau_scores, [2.5, 97.5])

    print(f"MAE          = {mae:.4f} (95% CI: {mae_ci[0]:.4f}–{mae_ci[1]:.4f})")
    print(f"MSE          = {mse:.4f} (95% CI: {mse_ci[0]:.4f}–{mse_ci[1]:.4f})")
    print(f"R²           = {r2:.4f}  (95% CI: {r2_ci[0]:.4f}–{r2_ci[1]:.4f})")
    print(f"Kendall's τ  = {tau:.4f} (95% CI: {tau_ci[0]:.4f}–{tau_ci[1]:.4f})")

    if save_file:
        results = {"MAE": mae, "MSE": mse, "R2": r2, "Kendall_tau": tau}
        with open(save_path, "w") as f:
            json.dump(results, f)
        print(f"Results saved to {save_path}")

    if show_plot:
        plt.figure(figsize=(6, 6))
        plt.scatter(y_test, y_pred, alpha=0.1, s=10)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", label="Perfect fit")
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title("Actual vs Predicted")
        plt.legend()
        plt.show()

In [ ]:
evaluate_model(y_test, y_pred, show_plot=True)